In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Training device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. We can prepare data, but training will be slower.")

PyTorch version: 2.11.0+cu130
Training device: cuda
GPU: Tesla T4


In [2]:
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd

# Download the dataset.
dataset_path = Path(
    kagglehub.dataset_download("sunnysai12345/news-summary")
)

# Some downloads contain ZIP archives. Extract them if needed.
csv_path = dataset_path / "news_summary_more.csv"

if not csv_path.exists():
    for archive in dataset_path.glob("*.zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dataset_path)

# Load the news text and headline pairs.
df = pd.read_csv(
    csv_path,
    encoding="utf-8",
    encoding_errors="strict",
)

print(f"Examples: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

display(df.head(3))

Using Colab cache for faster access to the 'news-summary' dataset.
Examples: 98,401
Columns: ['headlines', 'text']


,headlines,text
0,upGrad learner switches to career in ML & Al w...,"Saurav Kant, an alumnus of upGrad and IIIT-B's..."
1,Delhi techie wins free food from Swiggy for on...,Kunal Shah's credit card bill payment platform...
2,New Zealand end Rohit Sharma-led India's 12-ma...,New Zealand defeated India by 8 wickets in the...


In [3]:
%pip install -q ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [4]:
import html
import re
import unicodedata
import ftfy

def clean_text(value):
    if pd.isna(value):
        return ""

    text = html.unescape(str(value))
    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Keep df unchanged so we can compare against the original.
clean_df = df[["text", "headlines"]].copy()

for column in ["text", "headlines"]:
    clean_df[column] = clean_df[column].map(clean_text)

# Remove pairs with an empty input or target.
empty_mask = (
    clean_df["text"].eq("")
    | clean_df["headlines"].eq("")
)
empty_count = int(empty_mask.sum())
clean_df = clean_df.loc[~empty_mask].copy()

# Remove identical input–target pairs after cleaning.
duplicate_count = int(
    clean_df.duplicated(subset=["text", "headlines"]).sum()
)

clean_df = clean_df.drop_duplicates(
    subset=["text", "headlines"]
).reset_index(drop=True)

print(f"Original pairs:   {len(df):,}")
print(f"Empty removed:    {empty_count:,}")
print(f"Duplicates removed: {duplicate_count:,}")
print(f"Remaining pairs:  {len(clean_df):,}")

# Inspect a few examples changed by cleaning.
changed_mask = (
    df["text"].fillna("").ne(df["text"].map(clean_text))
    | df["headlines"].fillna("").ne(df["headlines"].map(clean_text))
)

comparison = pd.DataFrame({
    "original_text": df.loc[changed_mask, "text"],
    "cleaned_text": df.loc[changed_mask, "text"].map(clean_text),
    "original_headline": df.loc[changed_mask, "headlines"],
    "cleaned_headline": df.loc[changed_mask, "headlines"].map(clean_text),
})

display(comparison.head(5))

Original pairs:   98,401
Empty removed:    0
Duplicates removed: 27
Remaining pairs:  98,374


,original_text,cleaned_text,original_headline,cleaned_headline
3,"With Aegon Life iTerm Insurance plan, customer...","With Aegon Life iTerm Insurance plan, customer...",Aegon life iTerm insurance plan helps customer...,Aegon life iTerm insurance plan helps customer...
14,Reliance Industries' Chairman Mukesh Ambani's ...,Reliance Industries' Chairman Mukesh Ambani's ...,I only cried at my 'bidaai' as I felt peer pr...,I only cried at my 'bidaai' as I felt peer pre...
15,"Louis Vuitton owner LVMH, which makes high-end...","Louis Vuitton owner LVMH, which makes high-end...",Louis Vuitton owner to stockpile 4 months of w...,Louis Vuitton owner to stockpile 4 months of w...
16,Filmmaker Karan Johar and actress Tabu turned ...,Filmmaker Karan Johar and actress Tabu turned ...,"Karan Johar, Tabu turn showstoppers on opening...","Karan Johar, Tabu turn showstoppers on opening..."
17,"In a jibe at Congress President Rahul Gandhi, ...","In a jibe at Congress President Rahul Gandhi, ...",Those on bail will go to jail: PM Modi takes j...,Those on bail will go to jail: PM Modi takes j...


In [5]:
for index, row in comparison.head(2).iterrows():
    print(f"EXAMPLE {index}")

    print("\nORIGINAL TEXT:")
    print(row["original_text"])

    print("\nCLEANED TEXT:")
    print(row["cleaned_text"])

    print("\nORIGINAL HEADLINE:")
    print(row["original_headline"])

    print("\nCLEANED HEADLINE:")
    print(row["cleaned_headline"])

    print("\n" + "-" * 60 + "\n")

EXAMPLE 3

ORIGINAL TEXT:
With Aegon Life iTerm Insurance plan, customers can enjoy tax benefits on your premiums paid and save up to â¹46,800^ on taxes. The plan provides life cover up to the age of 100 years. Also, customers have options to insure against Critical Illnesses, Disability and Accidental Death Benefit Rider with a life cover up to the age of 80 years.

CLEANED TEXT:
With Aegon Life iTerm Insurance plan, customers can enjoy tax benefits on your premiums paid and save up to ₹46,800^ on taxes. The plan provides life cover up to the age of 100 years. Also, customers have options to insure against Critical Illnesses, Disability and Accidental Death Benefit Rider with a life cover up to the age of 80 years.

ORIGINAL HEADLINE:
Aegon life iTerm insurance plan helps customers save tax

CLEANED HEADLINE:
Aegon life iTerm insurance plan helps customers save tax

------------------------------------------------------------

EXAMPLE 14

ORIGINAL TEXT:
Reliance Industries' Chairman 

In [6]:
from sklearn.model_selection import GroupShuffleSplit

# First, reserve approximately 20% for validation and testing.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, heldout_idx = next(
    splitter.split(clean_df, groups=clean_df["text"])
)

train_df = clean_df.iloc[train_idx].reset_index(drop=True)
heldout_df = clean_df.iloc[heldout_idx].reset_index(drop=True)

# Divide the held-out portion into validation and test sets.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42,
)

val_idx, test_idx = next(
    splitter.split(heldout_df, groups=heldout_df["text"])
)

val_df = heldout_df.iloc[val_idx].reset_index(drop=True)
test_df = heldout_df.iloc[test_idx].reset_index(drop=True)

# Confirm that identical inputs don't cross split boundaries.
train_inputs = set(train_df["text"])
val_inputs = set(val_df["text"])
test_inputs = set(test_df["text"])

assert train_inputs.isdisjoint(val_inputs)
assert train_inputs.isdisjoint(test_inputs)
assert val_inputs.isdisjoint(test_inputs)

assert len(train_df) + len(val_df) + len(test_df) == len(clean_df)

for name, frame in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(
        f"{name}: {len(frame):,} pairs "
        f"({len(frame) / len(clean_df):.1%})"
    )

print("\nChecks passed: no identical inputs shared across sets.")

Training: 78,699 pairs (80.0%)
Validation: 9,838 pairs (10.0%)
Test: 9,837 pairs (10.0%)

Checks passed: no identical inputs shared across sets.


In [7]:
%pip install -q sentencepiece

In [8]:
from pathlib import Path
import sentencepiece as spm

# Check that our training data is still available.
assert "train_df" in globals(), (
    "Run the data loading, cleaning, and splitting cells first."
)

# Write training stories and headlines, one per line.
work_dir = Path("/content/seq2seq")
work_dir.mkdir(parents=True, exist_ok=True)

corpus_path = work_dir / "tokenizer_training.txt"

with corpus_path.open("w", encoding="utf-8") as f:
    for column in ["text", "headlines"]:
        for text in train_df[column]:
            f.write(text.replace("\n", " ").replace("\r", " ") + "\n")

# Learn the vocabulary using Unigram.
model_prefix = work_dir / "news_unigram_8000"

spm.SentencePieceTrainer.train(
    input=str(corpus_path),
    model_prefix=str(model_prefix),
    model_type="unigram",
    vocab_size=8000,
    character_coverage=1.0,
    normalization_rule_name="identity",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

# Load the trained tokenizer.
sp = spm.SentencePieceProcessor(
    model_file=str(model_prefix) + ".model"
)

print(f"\nVocabulary size: {sp.get_piece_size():,}")

for token_id in range(4):
    print(f"{token_id}: {sp.id_to_piece(token_id)}")


Vocabulary size: 8,000
0: <pad>
1: <unk>
2: <s>
3: </s>


In [9]:
example = train_df.iloc[0]

for label, text in [
    ("NEWS STORY", example["text"]),
    ("TARGET HEADLINE", example["headlines"]),
]:
    pieces = sp.encode_as_pieces(text)
    ids = sp.encode_as_ids(text)
    decoded = sp.decode_ids(ids)

    print(f"\n{label}")
    print("Original:", text)
    print("Pieces:", pieces)
    print("IDs:", ids)
    print("Decoded:", decoded)

    print(f"Words: {len(text.split())}")
    print(f"Tokens: {len(ids)}")
    print(f"Unknown tokens: {ids.count(sp.unk_id())}")
    print(f"Exact round trip: {decoded == text}")


NEWS STORY
Original: Saurav Kant, an alumnus of upGrad and IIIT-B's PG Program in Machine learning and Artificial Intelligence, was a Sr Systems Engineer at Infosys with almost 5 years of work experience. The program and upGrad's 360-degree career support helped him transition to a Data Scientist at Tech Mahindra with 90% salary hike. upGrad's Online Power Learning has powered 3 lakh+ careers.
Pieces: ['▁Sa', 'ur', 'av', '▁Kan', 't', ',', '▁an', '▁al', 'um', 'nu', 's', '▁of', '▁up', 'G', 'ra', 'd', '▁and', '▁II', 'IT', '-', 'B', "'", 's', '▁P', 'G', '▁Pro', 'gram', '▁in', '▁Machine', '▁learning', '▁and', '▁Ar', 'ti', 'fi', 'ci', 'al', '▁Intelligence', ',', '▁was', '▁a', '▁S', 'r', '▁System', 's', '▁E', 'ng', 'ine', 'er', '▁at', '▁Infosys', '▁with', '▁almost', '▁5', '▁years', '▁of', '▁work', '▁experience', '.', '▁The', '▁program', '▁and', '▁up', 'G', 'ra', 'd', "'", 's', '▁36', '0', '-', 'degree', '▁career', '▁support', '▁helped', '▁him', '▁transition', '▁to', '▁a', '▁Da', 'ta', '▁S', 

In [10]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Reuse encoded sequences if the earlier cell already finished.
if "train_source_ids" not in globals():
    train_source_ids = [
        sp.encode_as_ids(text)
        for text in tqdm(train_df["text"], desc="Encoding stories")
    ]

if "train_target_ids" not in globals():
    train_target_ids = [
        sp.encode_as_ids(text)
        for text in tqdm(train_df["headlines"], desc="Encoding headlines")
    ]

source_lengths = np.array([len(ids) for ids in train_source_ids])
target_lengths = np.array([len(ids) for ids in train_target_ids])

def summarize(lengths):
    return {
        "Median": np.median(lengths),
        "95th percentile": np.percentile(lengths, 95),
        "99th percentile": np.percentile(lengths, 99),
        "Maximum": lengths.max(),
    }

display(pd.DataFrame({
    "Story tokens": summarize(source_lengths),
    "Headline tokens": summarize(target_lengths),
}).round(1))

Encoding stories:   0%|          | 0/78699 [00:00<?, ?it/s]

Encoding headlines:   0%|          | 0/78699 [00:00<?, ?it/s]

,Story tokens,Headline tokens
Median,84.0,14.0
95th percentile,103.0,20.0
99th percentile,113.0,23.0
Maximum,149.0,31.0


In [11]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

PAD_ID = sp.pad_id()
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()

class NewsDataset(Dataset):
    def __init__(self, source_ids, target_ids):
        assert len(source_ids) == len(target_ids)
        self.source_ids = source_ids
        self.target_ids = target_ids

    def __len__(self):
        return len(self.source_ids)

    def __getitem__(self, index):
        return self.source_ids[index], self.target_ids[index]


def collate_batch(examples):
    sources = []
    decoder_inputs = []
    decoder_targets = []

    for source_ids, headline_ids in examples:
        sources.append(torch.tensor(source_ids, dtype=torch.long))

        decoder_inputs.append(torch.tensor(
            [BOS_ID] + headline_ids, dtype=torch.long
        ))

        decoder_targets.append(torch.tensor(
            headline_ids + [EOS_ID], dtype=torch.long
        ))

    # Preserve actual lengths so the encoder can skip padding later.
    source_lengths = torch.tensor(
        [len(source) for source in sources],
        dtype=torch.long,
    )

    return {
        "source": pad_sequence(
            sources, batch_first=True, padding_value=PAD_ID
        ),
        "source_lengths": source_lengths,
        "decoder_input": pad_sequence(
            decoder_inputs, batch_first=True, padding_value=PAD_ID
        ),
        "decoder_target": pad_sequence(
            decoder_targets, batch_first=True, padding_value=PAD_ID
        ),
    }


train_dataset = NewsDataset(train_source_ids, train_target_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(f"{name}: {tuple(tensor.shape)}")

source: (32, 109)
source_lengths: (32,)
decoder_input: (32, 20)
decoder_target: (32, 20)


In [12]:
input_ids = batch["decoder_input"][0].tolist()
target_ids = batch["decoder_target"][0].tolist()

for input_id, target_id in zip(input_ids, target_ids):
    if target_id == PAD_ID:
        break

    print(
        f"Receives: {sp.id_to_piece(input_id):15}"
        f"Should predict: {sp.id_to_piece(target_id)}"
    )

Receives: <s>            Should predict: ▁Mira
Receives: ▁Mira          Should predict: ▁Na
Receives: ▁Na            Should predict: ir
Receives: ir             Should predict: ▁wins
Receives: ▁wins          Should predict: ▁Director
Receives: ▁Director      Should predict: s
Receives: s              Should predict: ▁Award
Receives: ▁Award         Should predict: ▁at
Receives: ▁at            Should predict: ▁Women
Receives: ▁Women         Should predict: ▁In
Receives: ▁In            Should predict: ▁Film
Receives: ▁Film          Should predict: ▁Awards
Receives: ▁Awards        Should predict: </s>


In [13]:
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_id):
        super().__init__()

        # One learnable vector per vocabulary entry.
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)

        # Tell the LSTM where each story ends, excluding padding.
        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, cell) = self.lstm(packed)

        return hidden, cell


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = Encoder(
    vocab_size=sp.get_piece_size(),
    embedding_dim=128,
    hidden_dim=256,
    pad_id=PAD_ID,
).to(device)

# Inspect a forward pass; no training or weight updates yet.
with torch.no_grad():
    hidden, cell = encoder(
        batch["source"].to(device),
        batch["source_lengths"],
    )

print("Device:", device)
print("Embedding table:", tuple(encoder.embedding.weight.shape))
print("Hidden state:", tuple(hidden.shape))
print("Cell state:", tuple(cell.shape))

Device: cuda
Embedding table: (8000, 128)
Hidden state: (1, 32, 256)
Cell state: (1, 32, 256)


In [14]:
sentence = "the dog walks in the park"

token_ids = sp.encode(sentence, out_type=int)
pieces = [sp.id_to_piece(token_id) for token_id in token_ids]

# Start with no information carried from earlier tokens.
state = None

with torch.no_grad():
    for piece, token_id in zip(pieces, token_ids):
        # One story, one token.
        token = torch.tensor([[token_id]], device=device)

        # Look up this token's 128-number embedding.
        embedded = encoder.embedding(token)

        # Process this token using the previous hidden and cell states.
        # On the first step, None tells PyTorch to start both at zero.
        _, state = encoder.lstm(embedded, state)
        hidden, cell = state

        print(f"\nToken: {piece!r} | ID: {token_id}")
        print("Hidden, first 5:", hidden[0, 0, :5].cpu().tolist())
        print("Cell, first 5:  ", cell[0, 0, :5].cpu().tolist())


Token: '▁the' | ID: 5
Hidden, first 5: [0.0023222961463034153, -0.04571164399385452, 0.028085684403777122, -0.01730906218290329, -0.03639424219727516]
Cell, first 5:   [0.0038453072775155306, -0.14322395622730255, 0.10448840260505676, -0.027077829465270042, -0.06083131581544876]

Token: '▁dog' | ID: 1577
Hidden, first 5: [0.08229070901870728, -0.13351108133792877, -0.021186280995607376, -0.14992669224739075, -0.05734347552061081]
Cell, first 5:   [0.12374964356422424, -0.24321885406970978, -0.04969397187232971, -0.2584276497364044, -0.15788109600543976]

Token: '▁walk' | ID: 2619
Hidden, first 5: [-0.033370960503816605, -0.024421410635113716, 0.00962963979691267, 0.1282828450202942, 0.09402470290660858]
Cell, first 5:   [-0.05721447616815567, -0.04154170677065849, 0.019386278465390205, 0.19341573119163513, 0.18329684436321259]

Token: 's' | ID: 4
Hidden, first 5: [-0.1346920281648636, -0.11368754506111145, 0.12036475539207458, 0.0647238940000534, 0.12704327702522278]
Cell, first 5:   

In [15]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_id):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        # Convert each hidden state into one score per vocabulary token.
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, headline_input, hidden, cell):
        embedded = self.embedding(headline_input)

        # Start from the states supplied by the encoder.
        outputs, (hidden, cell) = self.lstm(
            embedded, (hidden, cell)
        )

        logits = self.output(outputs)

        return logits, hidden, cell


decoder = Decoder(
    vocab_size=sp.get_piece_size(),
    embedding_dim=128,
    hidden_dim=256,
    pad_id=PAD_ID,
).to(device)

In [16]:
print(decoder)

Decoder(
  (embedding): Embedding(8000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, batch_first=True)
  (output): Linear(in_features=256, out_features=8000, bias=True)
)


In [17]:
for name, value in batch.items():
    print(name, tuple(value.shape))

source (32, 109)
source_lengths (32,)
decoder_input (32, 20)
decoder_target (32, 20)


In [18]:
with torch.no_grad():
    # Read the 32 stories.
    encoder_hidden, encoder_cell = encoder(
        batch["source"].to(device),
        batch["source_lengths"],
    )

    # Use the story states while processing the headline inputs.
    logits, decoder_hidden, decoder_cell = decoder(
        batch["decoder_input"].to(device),
        encoder_hidden,
        encoder_cell,
    )

print("Decoder input:", tuple(batch["decoder_input"].shape))
print("Next-token scores:", tuple(logits.shape))
print("Targets:", tuple(batch["decoder_target"].shape))

Decoder input: (32, 20)
Next-token scores: (32, 20, 8000)
Targets: (32, 20)


In [19]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

targets = batch["decoder_target"].to(device)

loss = criterion(
    logits.reshape(-1, sp.get_piece_size()),
    targets.reshape(-1),
)

print("Initial loss:", loss.item())

Initial loss: 8.987710952758789


In [20]:
encoder.zero_grad()
decoder.zero_grad()

targets = batch["decoder_target"].to(device)

# Watch the output bias for one target token.
watched_id = targets[targets != PAD_ID][0].item()
bias_before = decoder.output.bias[watched_id].detach().item()

# 1. Forward pass.
encoder_hidden, encoder_cell = encoder(
    batch["source"].to(device),
    batch["source_lengths"],
)

logits, _, _ = decoder(
    batch["decoder_input"].to(device),
    encoder_hidden,
    encoder_cell,
)

# 2. Calculate loss.
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

loss = criterion(
    logits.reshape(-1, sp.get_piece_size()),
    targets.reshape(-1),
)

# 3. Calculate derivatives of loss with respect to parameters.
loss.backward()

bias_after = decoder.output.bias[watched_id].detach().item()
bias_gradient = decoder.output.bias.grad[watched_id].item()

print("Loss:", loss.item())
print("Watched token:", sp.id_to_piece(watched_id))
print("Bias before backward:", bias_before)
print("Bias after backward: ", bias_after)
print("Gradient:            ", bias_gradient)

Loss: 8.987710952758789
Watched token: ▁Mira
Bias before backward: -0.010614261031150818
Bias after backward:  -0.010614261031150818
Gradient:             -0.00208131386898458


In [21]:
print("Encoder device:", next(encoder.parameters()).device)
print("Decoder device:", next(decoder.parameters()).device)

gradient = decoder.output.bias.grad
print("Output bias gradients available:", gradient is not None)

if gradient is not None:
    print("Gradient norm:", gradient.norm().item())

Encoder device: cuda:0
Decoder device: cuda:0
Output bias gradients available: True
Gradient norm: 0.10411710292100906


In [22]:
parameters = list(encoder.parameters()) + list(decoder.parameters())

# Simple SGD makes the parameter update easy to inspect.
learning_rate = 0.1
optimizer = torch.optim.SGD(parameters, lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

source = batch["source"].to(device)
decoder_input = batch["decoder_input"].to(device)
targets = batch["decoder_target"].to(device)

encoder.train()
decoder.train()
optimizer.zero_grad()

# 1. Forward pass and loss.
hidden, cell = encoder(source, batch["source_lengths"])
logits, _, _ = decoder(decoder_input, hidden, cell)

loss_before = criterion(
    logits.reshape(-1, sp.get_piece_size()),
    targets.reshape(-1),
)

# 2. Calculate fresh gradients.
loss_before.backward()

# Inspect one parameter before updating.
watched_id = targets[targets != PAD_ID][0].item()
bias_before = decoder.output.bias[watched_id].detach().item()
gradient = decoder.output.bias.grad[watched_id].item()

expected_bias = bias_before - learning_rate * gradient

# 3. Update parameters in both models.
optimizer.step()

bias_after = decoder.output.bias[watched_id].detach().item()

# 4. Measure loss again using the updated parameters.
with torch.no_grad():
    hidden, cell = encoder(source, batch["source_lengths"])
    updated_logits, _, _ = decoder(decoder_input, hidden, cell)

    loss_after = criterion(
        updated_logits.reshape(-1, sp.get_piece_size()),
        targets.reshape(-1),
    )

print("Watched token:", sp.id_to_piece(watched_id))
print("Bias before:  ", bias_before)
print("Gradient:     ", gradient)
print("Expected bias:", expected_bias)
print("Actual bias:  ", bias_after)
print()
print("Loss before:", loss_before.item())
print("Loss after: ", loss_after.item())
print("Loss change:", loss_after.item() - loss_before.item())

Watched token: ▁Mira
Bias before:   -0.010614261031150818
Gradient:      -0.00208131386898458
Expected bias: -0.01040612964425236
Actual bias:   -0.010406129993498325

Loss before: 8.987710952758789
Loss after:  8.984949111938477
Loss change: -0.0027618408203125


In [23]:
from torch.utils.data import DataLoader

for name, obj in list(globals().items()):
    if isinstance(obj, DataLoader):
        print(
            name,
            "| examples:", len(obj.dataset),
            "| batches:", len(obj),
            "| batch size:", obj.batch_size,
        )

train_loader | examples: 78699 | batches: 2460 | batch size: 32


In [24]:
# Show likely validation objects without printing their contents.
for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if "val" in name.lower() or "valid" in name.lower():
        try:
            size = len(obj)
        except TypeError:
            size = "not applicable"

        print(name, "| type:", type(obj).__name__, "| length:", size)

print("\nTraining dataset type:", type(train_loader.dataset).__name__)
print("Batching function:", train_loader.collate_fn)

val_idx | type: ndarray | length: 9838
val_df | type: DataFrame | length: 9838
val_inputs | type: set | length: 9834
value | type: Tensor | length: 32

Training dataset type: NewsDataset
Batching function: <function collate_batch at 0x79607d82a480>


In [26]:
import inspect

print("Constructor:", inspect.signature(NewsDataset.__init__))
print("Stored attributes:", list(vars(train_loader.dataset).keys()))
print("Validation columns:", val_df.columns.tolist())

Constructor: (self, source_ids, target_ids)
Stored attributes: ['source_ids', 'target_ids']
Validation columns: ['text', 'headlines']


In [27]:
val_dataset = NewsDataset(
    [sp.encode(text, out_type=int) for text in val_df["text"]],
    [sp.encode(text, out_type=int) for text in val_df["headlines"]],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch,
)

check = next(iter(val_loader))

assert (check["decoder_input"][:, 0] == sp.bos_id()).all(), \
    "Decoder inputs are missing their start token."

for target in check["decoder_target"]:
    real_tokens = target[target != PAD_ID]
    assert real_tokens[-1].item() == sp.eos_id(), \
        "Decoder targets are missing their end token."

print("Validation examples:", len(val_dataset))
print("Validation batches:", len(val_loader))

Validation examples: 9838
Validation batches: 308


In [28]:
import time
from pathlib import Path
from tqdm.auto import tqdm

parameters = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.Adam(parameters, lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# A separate directory for this run.
run_dir = Path("checkpoints") / time.strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)

# Save the exact tokenizer alongside the model.
(run_dir / "tokenizer.model").write_bytes(sp.serialized_model_proto())

def run_epoch(loader, training):
    encoder.train(training)
    decoder.train(training)

    total_loss = 0.0
    total_tokens = 0

    progress = tqdm(
        loader,
        desc="Training" if training else "Validation",
    )

    with torch.set_grad_enabled(training):
        for current_batch in progress:
            source = current_batch["source"].to(device)
            inputs = current_batch["decoder_input"].to(device)
            targets = current_batch["decoder_target"].to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            # Fresh encoder states for every batch.
            hidden, cell = encoder(
                source,
                current_batch["source_lengths"],
            )

            logits, _, _ = decoder(inputs, hidden, cell)

            loss = criterion(
                logits.reshape(-1, sp.get_piece_size()),
                targets.reshape(-1),
            )

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss; stopping this run.")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    parameters, max_norm=1.0,
                    error_if_nonfinite=True,
                )
                optimizer.step()

            # Weight the epoch average by non-padding target tokens.
            token_count = (targets != PAD_ID).sum().item()
            total_loss += loss.item() * token_count
            total_tokens += token_count

            progress.set_postfix(
                loss=f"{total_loss / total_tokens:.4f}"
            )

    return total_loss / total_tokens


started = time.perf_counter()

train_loss = run_epoch(train_loader, training=True)
val_loss = run_epoch(val_loader, training=False)

checkpoint_path = run_dir / "epoch-01.pt"

torch.save({
    "epoch": 1,
    "encoder": encoder.state_dict(),
    "decoder": decoder.state_dict(),
    "optimizer": optimizer.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
    "config": {
        "vocab_size": sp.get_piece_size(),
        "embedding_dim": 128,
        "hidden_dim": 256,
        "pad_id": PAD_ID,
    },
}, checkpoint_path)

print(f"\nTraining loss:   {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Elapsed minutes: {(time.perf_counter() - started) / 60:.1f}")
print("Checkpoint:", checkpoint_path)

Training:   0%|          | 0/2460 [00:00<?, ?it/s]

Validation:   0%|          | 0/308 [00:00<?, ?it/s]


Training loss:   5.9019
Validation loss: 5.2850
Elapsed minutes: 0.7
Checkpoint: checkpoints/20261019-121050/epoch-01.pt


In [29]:
def generate_headline(text, max_new_tokens=40):
    encoder.eval()
    decoder.eval()

    source_ids = sp.encode(text, out_type=int)

    if not source_ids:
        return ""

    source = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )
    lengths = torch.tensor([len(source_ids)], dtype=torch.long)

    generated_ids = []

    with torch.no_grad():
        hidden, cell = encoder(source, lengths)

        # Start the headline with the beginning-of-sequence token.
        token = torch.tensor(
            [[sp.bos_id()]], dtype=torch.long, device=device
        )

        for _ in range(max_new_tokens):
            logits, hidden, cell = decoder(token, hidden, cell)

            scores = logits[0, -1].clone()
            scores[PAD_ID] = -torch.inf
            scores[sp.bos_id()] = -torch.inf

            # Greedy decoding: choose the highest-scoring token.
            next_id = scores.argmax().item()

            if next_id == sp.eos_id():
                break

            generated_ids.append(next_id)
            token = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    return sp.decode(generated_ids)


# Use validation examples; keep the test split reserved.
for index in range(3):
    row = val_df.iloc[index]

    print("STORY:", row["text"])
    print("REFERENCE:", row["headlines"])
    print("GENERATED:", generate_headline(row["text"]))
    print()

STORY: Speaking about the sexual harassment allegations against Rajkumar Hirani, Sonam Kapoor said, "I've known Hirani for many years...What if it's not true, the [#MeToo] movement will get derailed." "In the #MeToo movement, I always believe a woman. But in this case, we need to reserve our judgment," she added. Hirani has been accused by an assistant who worked in 'Sanju'.
REFERENCE: Have known Hirani for yrs, what if MeToo claims are not true: Sonam
GENERATED: I'm not a big: Rahul on 'Bama' hits theatres

STORY: Filmmaker Karan Johar and actress Tabu turned showstoppers for Gaurav Gupta on the opening night of Lakmé Fashion Week Summer/ Resort 2019. While Johar wore a red sequinned jacket with black pants, Tabu walked the ramp in a grey embellished gown. The fashion show, which began on January 29, will continue till February 3.
REFERENCE: Karan Johar, Tabu turn showstoppers on opening night of LFW
GENERATED: India's first-ever stalker, says 'Mama' hits the theatres

STORY: Days aft

In [32]:
best_val_loss = val_loss
best_checkpoint_path = checkpoint_path
epochs_without_improvement = 0

for epoch in range(6, 11):
    print(f"\nEpoch {epoch}")

    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    checkpoint_path = run_dir / f"epoch-{epoch:02d}.pt"

    torch.save({
        "epoch": epoch,
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "optimizer": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "config": {
            "vocab_size": sp.get_piece_size(),
            "embedding_dim": 128,
            "hidden_dim": 256,
            "pad_id": PAD_ID,
        },
    }, checkpoint_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_checkpoint_path = checkpoint_path
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"Training loss:   {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")
    print("Best checkpoint:", best_checkpoint_path)

    # Generate from the same validation story for comparison.
    print("Sample:", generate_headline(val_df.iloc[0]["text"]))

    if epochs_without_improvement >= 2:
        print("Stopping: validation loss did not improve for two epochs.")
        break


Epoch 6


Training:   0%|          | 0/2460 [00:00<?, ?it/s]

Validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   3.7039
Validation loss: 4.3206
Best checkpoint: checkpoints/20261019-121050/epoch-06.pt
Sample: I'll be a superstar: Kangana on 'Sanju' featuring Salman Khan

Epoch 7


Training:   0%|          | 0/2460 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   3.3931
Validation loss: 4.3071
Best checkpoint: checkpoints/20261019-121050/epoch-07.pt
Sample: If I was asked to do a film: Producer on 'Sanju' director

Epoch 9


Training:   0%|          | 0/2460 [00:00<?, ?it/s]

Validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   3.2663
Validation loss: 4.3222
Best checkpoint: checkpoints/20261019-121050/epoch-07.pt
Sample: I'll be ashamed to work: Ayushmann on 'Koffee With Karan'
Stopping: validation loss did not improve for two epochs.


In [31]:
from itertools import islice

encoder.eval()
decoder.eval()

totals = {"correct_story": 0.0, "wrong_story": 0.0}
token_total = 0

with torch.no_grad():
    for current_batch in islice(val_loader, 10):
        source = current_batch["source"].to(device)
        inputs = current_batch["decoder_input"].to(device)
        targets = current_batch["decoder_target"].to(device)

        hidden, cell = encoder(
            source,
            current_batch["source_lengths"],
        )

        # Keep the headline inputs identical in both conditions.
        correct_logits, _, _ = decoder(inputs, hidden, cell)

        # Shift states across examples: each headline gets another story.
        wrong_logits, _, _ = decoder(
            inputs,
            hidden.roll(shifts=1, dims=1),
            cell.roll(shifts=1, dims=1),
        )

        count = (targets != PAD_ID).sum().item()
        token_total += count

        for name, scores in [
            ("correct_story", correct_logits),
            ("wrong_story", wrong_logits),
        ]:
            loss = criterion(
                scores.reshape(-1, sp.get_piece_size()),
                targets.reshape(-1),
            )
            totals[name] += loss.item() * count

correct_loss = totals["correct_story"] / token_total
wrong_loss = totals["wrong_story"] / token_total

print("Correct-story loss:", correct_loss)
print("Wrong-story loss:  ", wrong_loss)
print("Difference:        ", wrong_loss - correct_loss)

Correct-story loss: 4.348570647069808
Wrong-story loss:   5.360330041964228
Difference:         1.01175939489442


In [33]:
best_checkpoint_path = (
    run_dir / "epoch-07.pt"
)

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True,
)

encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

encoder.eval()
decoder.eval()

print("Loaded epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

Loaded epoch: 7
Validation loss: 4.301907780499844


In [34]:
sample_rows = val_df.sample(n=5, random_state=42)

for _, row in sample_rows.iterrows():
    print("STORY:", row["text"])
    print("REFERENCE:", row["headlines"])
    print("GENERATED:", generate_headline(row["text"]))
    print()

STORY: The farmers at Nindar village near Jaipur in Rajasthan have been burying themselves in mud every day since October 2 to protest against land acquisition for housing projects. The Jaipur Development Authority had allegedly acquired 1300 acres of land from farmers in 2011 and since then they have been protesting against it in various ways.
REFERENCE: Raj farmers half-bury themselves to protest land acquisition
GENERATED: 2 killed, 2 injured in suicide attack in J&K

STORY: An Air India flight attendant was allegedly slapped by the cabin supervisor last week for serving a non-veg platter to a vegetarian passenger. The attendant had mistakenly served the wrong platter to a business-class passenger, following which the passenger called the supervisor. After the attendant apologised and replaced the food, the supervisor took her to the side and slapped her.
REFERENCE: Supervisor slaps air hostess for serving wrong meal to flyer
GENERATED: Air India flight makes emergency landing for 1

In [35]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    str(run_dir),
    "zip",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
from torch.nn.utils.rnn import pad_packed_sequence

class AttentionEncoder(Encoder):
    def forward(self, source, source_lengths):
        embedded = self.embedding(source)

        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)

        # Recover one 256-number output per story-token position.
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source.size(1),
        )

        return outputs, hidden, cell


attention_encoder = AttentionEncoder(
    vocab_size=sp.get_piece_size(),
    embedding_dim=128,
    hidden_dim=256,
    pad_id=PAD_ID,
).to(device)

# Copy the baseline encoder's learned parameters.
attention_encoder.load_state_dict(encoder.state_dict())

with torch.no_grad():
    encoder_outputs, hidden, cell = attention_encoder(
        batch["source"].to(device),
        batch["source_lengths"],
    )

print("All encoder outputs:", tuple(encoder_outputs.shape))
print("Final hidden state:", tuple(hidden.shape))
print("Final cell state:", tuple(cell.shape))

All encoder outputs: (32, 109, 256)
Final hidden state: (1, 32, 256)
Final cell state: (1, 32, 256)


In [37]:
class AttentionDecoder(Decoder):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_id):
        super().__init__(
            vocab_size, embedding_dim, hidden_dim, pad_id
        )

        # Transform decoder outputs before comparing with story outputs.
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Turn story context into an adjustment to decoder outputs.
        self.context_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )

        self.scale = hidden_dim ** 0.5

        # Initially preserve baseline predictions.
        # Training will learn how to use the added context.
        nn.init.zeros_(self.context_projection.weight)

    def forward(
        self, headline_input, hidden, cell,
        encoder_outputs, source_mask,
    ):
        embedded = self.embedding(headline_input)

        decoder_outputs, (hidden, cell) = self.lstm(
            embedded, (hidden, cell)
        )

        # Compare each headline position with every story position.
        queries = self.query(decoder_outputs)

        scores = torch.bmm(
            queries,
            encoder_outputs.transpose(1, 2),
        ) / self.scale

        # Padding must receive no attention.
        scores = scores.masked_fill(
            ~source_mask.unsqueeze(1), -torch.inf
        )

        attention_weights = torch.softmax(scores, dim=-1)

        # Weighted sum of story outputs for each headline position.
        context = torch.bmm(
            attention_weights, encoder_outputs
        )

        combined = (
            decoder_outputs + self.context_projection(context)
        )
        logits = self.output(combined)

        return logits, hidden, cell, attention_weights


attention_decoder = AttentionDecoder(
    vocab_size=sp.get_piece_size(),
    embedding_dim=128,
    hidden_dim=256,
    pad_id=PAD_ID,
).to(device)

# Copy the existing embedding, LSTM, and vocabulary output layer.
attention_decoder.embedding.load_state_dict(
    decoder.embedding.state_dict()
)
attention_decoder.lstm.load_state_dict(
    decoder.lstm.state_dict()
)
attention_decoder.output.load_state_dict(
    decoder.output.state_dict()
)

with torch.no_grad():
    source = batch["source"].to(device)
    inputs = batch["decoder_input"].to(device)
    lengths = batch["source_lengths"]

    # True for real story positions, False for padding.
    source_mask = (
        torch.arange(source.size(1), device=device)[None, :]
        < lengths.to(device)[:, None]
    )

    story_outputs, hidden, cell = attention_encoder(
        source, lengths
    )

    attention_logits, _, _, weights = attention_decoder(
        inputs, hidden, cell,
        story_outputs, source_mask,
    )

print("Token scores:", tuple(attention_logits.shape))
print("Attention weights:", tuple(weights.shape))
print(
    "Weight sum for first prediction:",
    weights[0, 0].sum().item(),
)

Token scores: (32, 20, 8000)
Attention weights: (32, 20, 109)
Weight sum for first prediction: 0.9999999403953552


In [38]:
import time
from pathlib import Path
from tqdm.auto import tqdm

attention_parameters = (
    list(attention_encoder.parameters())
    + list(attention_decoder.parameters())
)

attention_optimizer = torch.optim.Adam(
    attention_parameters, lr=0.001
)

attention_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

attention_run_dir = (
    Path("checkpoints")
    / ("attention-" + time.strftime("%Y%m%d-%H%M%S"))
)
attention_run_dir.mkdir(parents=True, exist_ok=True)

(attention_run_dir / "tokenizer.model").write_bytes(
    sp.serialized_model_proto()
)

def run_attention_epoch(loader, training):
    attention_encoder.train(training)
    attention_decoder.train(training)

    total_loss = 0.0
    total_tokens = 0

    progress = tqdm(
        loader,
        desc="Attention training" if training else "Attention validation",
    )

    with torch.set_grad_enabled(training):
        for current_batch in progress:
            source = current_batch["source"].to(device)
            lengths = current_batch["source_lengths"]
            inputs = current_batch["decoder_input"].to(device)
            targets = current_batch["decoder_target"].to(device)

            source_mask = (
                torch.arange(source.size(1), device=device)[None, :]
                < lengths.to(device)[:, None]
            )

            if training:
                attention_optimizer.zero_grad(set_to_none=True)

            story_outputs, hidden, cell = attention_encoder(
                source, lengths
            )

            logits, _, _, _ = attention_decoder(
                inputs, hidden, cell,
                story_outputs, source_mask,
            )

            loss = attention_criterion(
                logits.reshape(-1, sp.get_piece_size()),
                targets.reshape(-1),
            )

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss; stopping.")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    attention_parameters,
                    max_norm=1.0,
                    error_if_nonfinite=True,
                )
                attention_optimizer.step()

            count = (targets != PAD_ID).sum().item()
            total_loss += loss.item() * count
            total_tokens += count

            progress.set_postfix(
                loss=f"{total_loss / total_tokens:.4f}"
            )

    return total_loss / total_tokens


# Evaluate before adding any attention training.
attention_initial_val = run_attention_epoch(
    val_loader, training=False
)

attention_train_loss = run_attention_epoch(
    train_loader, training=True
)

attention_val_loss = run_attention_epoch(
    val_loader, training=False
)

attention_checkpoint_path = attention_run_dir / "epoch-01.pt"

torch.save({
    "epoch": 1,
    "encoder": attention_encoder.state_dict(),
    "decoder": attention_decoder.state_dict(),
    "optimizer": attention_optimizer.state_dict(),
    "train_loss": attention_train_loss,
    "val_loss": attention_val_loss,
    "config": {
        "architecture": "lstm_scaled_dot_attention_residual",
        "vocab_size": sp.get_piece_size(),
        "embedding_dim": 128,
        "hidden_dim": 256,
        "pad_id": PAD_ID,
    },
}, attention_checkpoint_path)

print("\nValidation before:", attention_initial_val)
print("Training loss:    ", attention_train_loss)
print("Validation after: ", attention_val_loss)
print("Checkpoint:       ", attention_checkpoint_path)

Attention validation:   0%|          | 0/308 [00:00<?, ?it/s]

Attention training:   0%|          | 0/2460 [00:00<?, ?it/s]

Attention validation:   0%|          | 0/308 [00:00<?, ?it/s]


Validation before: 4.301907780499844
Training loss:     3.3027451535836017
Validation after:  3.964056722566886
Checkpoint:        checkpoints/attention-20261019-123539/epoch-01.pt


In [39]:
def generate_attention_headline(text, max_new_tokens=40):
    attention_encoder.eval()
    attention_decoder.eval()

    ids = sp.encode(text, out_type=int)
    if not ids:
        return ""

    source = torch.tensor(
        [ids], dtype=torch.long, device=device
    )
    lengths = torch.tensor([len(ids)], dtype=torch.long)
    source_mask = torch.ones_like(source, dtype=torch.bool)

    generated_ids = []

    with torch.no_grad():
        story_outputs, hidden, cell = attention_encoder(
            source, lengths
        )

        token = torch.tensor(
            [[sp.bos_id()]], dtype=torch.long, device=device
        )

        for _ in range(max_new_tokens):
            logits, hidden, cell, _ = attention_decoder(
                token, hidden, cell,
                story_outputs, source_mask,
            )

            scores = logits[0, -1].clone()
            scores[PAD_ID] = -torch.inf
            scores[sp.bos_id()] = -torch.inf

            next_id = scores.argmax().item()

            if next_id == sp.eos_id():
                break

            generated_ids.append(next_id)
            token = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    return sp.decode(generated_ids)


# Same five validation stories as before.
for _, row in val_df.sample(n=5, random_state=42).iterrows():
    print("STORY:", row["text"])
    print("REFERENCE:", row["headlines"])
    print("BASELINE:", generate_headline(row["text"]))
    print("ATTENTION:", generate_attention_headline(row["text"]))
    print()

STORY: The farmers at Nindar village near Jaipur in Rajasthan have been burying themselves in mud every day since October 2 to protest against land acquisition for housing projects. The Jaipur Development Authority had allegedly acquired 1300 acres of land from farmers in 2011 and since then they have been protesting against it in various ways.
REFERENCE: Raj farmers half-bury themselves to protest land acquisition
BASELINE: 2 killed, 2 injured in suicide attack in J&K
ATTENTION: 93 people who missed penalties for 4 months in Rajasthan

STORY: An Air India flight attendant was allegedly slapped by the cabin supervisor last week for serving a non-veg platter to a vegetarian passenger. The attendant had mistakenly served the wrong platter to a business-class passenger, following which the passenger called the supervisor. After the attendant apologised and replaced the food, the supervisor took her to the side and slapped her.
REFERENCE: Supervisor slaps air hostess for serving wrong meal

In [40]:
attention_best_val = attention_val_loss
attention_best_path = attention_checkpoint_path
attention_stale_epochs = 0

for epoch in range(2, 6):
    print(f"\nAttention epoch {epoch}")

    attention_train_loss = run_attention_epoch(
        train_loader, training=True
    )
    attention_val_loss = run_attention_epoch(
        val_loader, training=False
    )

    path = attention_run_dir / f"epoch-{epoch:02d}.pt"

    torch.save({
        "epoch": epoch,
        "encoder": attention_encoder.state_dict(),
        "decoder": attention_decoder.state_dict(),
        "optimizer": attention_optimizer.state_dict(),
        "train_loss": attention_train_loss,
        "val_loss": attention_val_loss,
        "config": {
            "architecture": "lstm_scaled_dot_attention_residual",
            "vocab_size": sp.get_piece_size(),
            "embedding_dim": 128,
            "hidden_dim": 256,
            "pad_id": PAD_ID,
        },
    }, path)

    if attention_val_loss < attention_best_val:
        attention_best_val = attention_val_loss
        attention_best_path = path
        attention_stale_epochs = 0
    else:
        attention_stale_epochs += 1

    print(f"Training loss:   {attention_train_loss:.4f}")
    print(f"Validation loss: {attention_val_loss:.4f}")
    print("Best checkpoint:", attention_best_path)
    print(
        "Air India sample:",
        generate_attention_headline(
            val_df.sample(n=5, random_state=42).iloc[1]["text"]
        ),
    )

    if attention_stale_epochs >= 2:
        print("Stopping: no validation improvement for two epochs.")
        break


Attention epoch 2


Attention training:   0%|          | 0/2460 [00:00<?, ?it/s]

Attention validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   2.7653
Validation loss: 3.5330
Best checkpoint: checkpoints/attention-20261019-123539/epoch-02.pt
Air India sample: Air India flight caught on flight, stops vegetarian

Attention epoch 3


Attention training:   0%|          | 0/2460 [00:00<?, ?it/s]

...

Attention validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   2.0163
Validation loss: 3.2375
Best checkpoint: checkpoints/attention-20261019-123539/epoch-04.pt
Air India sample: Air India slaps man for not letting passenger on flight

Attention epoch 5


Attention training:   0%|          | 0/2460 [00:00<?, ?it/s]

Attention validation:   0%|          | 0/308 [00:00<?, ?it/s]

Training loss:   1.8055
Validation loss: 3.2212
Best checkpoint: checkpoints/attention-20261019-123539/epoch-05.pt
Air India sample: Air India slaps female passenger, claims Air India staffer slapped


In [41]:
for _, row in val_df.sample(n=5, random_state=42).iterrows():
    print("STORY:", row["text"])
    print("REFERENCE:", row["headlines"])
    print("BASELINE:", generate_headline(row["text"]))
    print("ATTENTION:", generate_attention_headline(row["text"]))
    print()

STORY: The farmers at Nindar village near Jaipur in Rajasthan have been burying themselves in mud every day since October 2 to protest against land acquisition for housing projects. The Jaipur Development Authority had allegedly acquired 1300 acres of land from farmers in 2011 and since then they have been protesting against it in various ways.
REFERENCE: Raj farmers half-bury themselves to protest land acquisition
BASELINE: 2 killed, 2 injured in suicide attack in J&K
ATTENTION: Jaipur farmers protest over stft in Rajasthan

STORY: An Air India flight attendant was allegedly slapped by the cabin supervisor last week for serving a non-veg platter to a vegetarian passenger. The attendant had mistakenly served the wrong platter to a business-class passenger, following which the passenger called the supervisor. After the attendant apologised and replaced the food, the supervisor took her to the side and slapped her.
REFERENCE: Supervisor slaps air hostess for serving wrong meal to flyer
B

In [42]:
def inspect_attention(text, max_new_tokens=40):
    attention_encoder.eval()
    attention_decoder.eval()

    source_ids = sp.encode(text, out_type=int)
    source_pieces = [sp.id_to_piece(i) for i in source_ids]

    if not source_ids:
        return "", None

    source = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )
    lengths = torch.tensor([len(source_ids)], dtype=torch.long)
    source_mask = torch.ones_like(source, dtype=torch.bool)

    generated_ids = []
    attention_rows = []

    with torch.no_grad():
        story_outputs, hidden, cell = attention_encoder(
            source, lengths
        )

        token = torch.tensor(
            [[sp.bos_id()]], dtype=torch.long, device=device
        )

        for _ in range(max_new_tokens):
            logits, hidden, cell, weights = attention_decoder(
                token, hidden, cell,
                story_outputs, source_mask,
            )

            scores = logits[0, -1].clone()
            scores[PAD_ID] = -torch.inf
            scores[sp.bos_id()] = -torch.inf

            next_id = scores.argmax().item()

            if next_id == sp.eos_id():
                break

            generated_ids.append(next_id)
            attention_rows.append(weights[0, -1].cpu())

            token = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    headline = sp.decode(generated_ids)
    print("GENERATED:", headline)
    print("\nAttention used to predict each token:\n")

    for token_id, row in zip(generated_ids, attention_rows):
        values, positions = row.topk(min(3, len(source_ids)))

        strongest = []
        for weight, position in zip(values.tolist(), positions.tolist()):
            # Include nearby pieces to help identify the source phrase.
            start = max(0, position - 2)
            end = min(len(source_ids), position + 3)
            nearby = sp.decode(source_ids[start:end])

            strongest.append(
                f"{position}: {source_pieces[position]!r} "
                f"({weight:.1%}; nearby: {nearby!r})"
            )

        print(
            f"{sp.id_to_piece(token_id)!r} → "
            + " | ".join(strongest)
        )

    if not attention_rows:
        return headline, None

    matrix = torch.stack(attention_rows).numpy()
    return headline, (source_pieces, generated_ids, matrix)


# Select the same Andhra story from our five-example comparison.
andhra_row = val_df.sample(n=5, random_state=42).iloc[2]

print("STORY:", andhra_row["text"])
print("REFERENCE:", andhra_row["headlines"])
print()

attention_headline, attention_trace = inspect_attention(
    andhra_row["text"]
)

STORY: Aiming to end VIP culture in the state, the Andhra Pradesh government on Saturday banned the use of blue beacons and hooters on cars belonging to dignitaries. According to the new order, only emergency and disaster management vehicles will be permitted to use blue beacons. These beacons are generally used by police and IAS officers.
REFERENCE: Andhra bans use of blue beacons, hooters by officials

GENERATED: Andhra govt bans use of blue beacon, homsers on cars

Attention used to predict each token:

'▁Andhra' → 14: '▁Andhra' (27.5%; nearby: ', the Andhra Pradesh government') | 15: '▁Pradesh' (14.1%; nearby: 'the Andhra Pradesh government on') | 5: '▁V' (8.5%; nearby: 'to end VIP')
'▁govt' → 16: '▁government' (32.9%; nearby: 'Andhra Pradesh government on Saturday') | 15: '▁Pradesh' (18.4%; nearby: 'the Andhra Pradesh government on') | 18: '▁Saturday' (11.5%; nearby: 'government on Saturday banned the')
'▁ban' → 19: '▁banned' (32.8%; nearby: 'on Saturday banned the use') | 18: '▁S